In [ ]:
import os
import subprocess
import sys
import site
import importlib

# FIRST - environment variables
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
*
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas"
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas-blackwell"

# SECOND - copy nvidia utilities and set permissions
_util = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script"
if os.path.exists(_util):
    # Copy files
    subprocess.run(f"tar --no-same-permissions -cf - -C {_util} . | tar -xf - -C /tmp", shell=True, check=False)
    # Make all triton files executable (including ptxas-blackwell)
    subprocess.run("chmod +x /tmp/triton/backends/nvidia/bin/ptxas*", shell=True, check=False)
    subprocess.run("chmod +x /tmp/triton/backends/nvidia/bin/triton*", shell=True, check=False)
    # Add python packages to path
    site.addsitedir(f"{_util}/nvidia_cutlass_dsl/python_packages/")
sys.path.insert(0, '/tmp')

# THIRD - install missing packages
_packages = {
    "unsloth": "/kaggle/input/nemotron-packages/packages/unsloth*.whl",
    "trl": "/kaggle/input/nemotron-packages/packages/trl*.whl",
    "peft": "/kaggle/input/nemotron-packages/packages/peft*.whl",
    "transformers": "/kaggle/input/nemotron-packages/packages/transformers*.whl",
    "datasets": "/kaggle/input/nemotron-packages/packages/datasets*.whl",
    "accelerate": "/kaggle/input/nemotron-packages/packages/accelerate*.whl",
    "bitsandbytes": "/kaggle/input/nemotron-packages/packages/bitsandbytes*.whl",
    "causal_conv1d": "/kaggle/input/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
    "mamba_ssm": "/kaggle/input/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
}
for _pkg, _wheel in _packages.items():
    if not importlib.util.find_spec(_pkg.replace("-", "_")):
        subprocess.run(f"pip install -q --no-index --find-links /kaggle/input/nemotron-packages/packages {_wheel}", shell=True)

# FOURTH - imports
import pandas as pd
import torch
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
import kagglehub

In [ ]:
import polars as pl
train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')
train.head()

In [11]:
# import os
# import subprocess
# import sys

# # FIRST — set env
# os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas"
# os.environ["TRANSFORMERS_NO_TF"] = "1"
# os.environ["TRANSFORMERS_NO_FLAX"] = "1"
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# # SECOND — copy and chmod
# subprocess.run(
#     "tar --no-same-permissions -cf - -C /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script . | tar -xf - -C /tmp",
#     shell=True, check=False
# )
# subprocess.run(
#     "find /tmp/triton -type f | xargs chmod +x",
#     shell=True, check=False
# )
# sys.path.insert(0, '/tmp')

# # THIRD — installs
# subprocess.run(
#     "pip install -q --no-index --find-links /kaggle/input/nemotron-packages/packages "
#     "unsloth trl peft transformers datasets accelerate bitsandbytes",
#     shell=True
# )
# subprocess.run(
#     "pip install -q /kaggle/input/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
#     shell=True
# )
# subprocess.run(
#     "pip install -q /kaggle/input/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
#     shell=True
# )

ERROR: Could not find a version that satisfies the requirement unsloth (from versions: none)
ERROR: No matching distribution found for unsloth
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl'

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl'



In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import site, subprocess, importlib.util, sys
from pathlib import Path

# # ── FIRST: env ────────────────────────────────────────────────────────────────
# os.environ.update({
#     "TRANSFORMERS_NO_TF": "1",
#     "TRANSFORMERS_NO_FLAX": "1",
#     "CUDA_VISIBLE_DEVICES": "0",
#     "TRITON_PTXAS_PATH": "/tmp/triton/backends/nvidia/bin/ptxas",
#     "TRITON_PTXAS_BLACKWELL_PATH": "/tmp/triton/backends/nvidia/bin/ptxas-blackwell",  # ← ADD
# })
# # ── SECOND: copy + chmod ──────────────────────────────────────────────────────
# _util = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script"
# if os.path.exists(_util):
#     subprocess.run(f"tar --no-same-permissions -cf - -C {_util} . | tar -xf - -C /tmp", shell=True, check=False)
#     subprocess.run("find /tmp/triton -type f | xargs chmod +x", shell=True, check=False)
#     site.addsitedir(f"{_util}/nvidia_cutlass_dsl/python_packages/")
# sys.path.insert(0, '/tmp')

# # ── THIRD: imports ────────────────────────────────────────────────────────────
# import kagglehub
# import pandas as pd
# import torch
# from peft import LoraConfig, TaskType, get_peft_model
# from torch.utils.data import Dataset
# from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

# try:
#     import mamba_ssm  # noqa: F401
# except ImportError:
#     pass

# ── config ────────────────────────────────────────────────────────────────────
MODEL_PATH      = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
DATA_DIR        = Path("/kaggle/input/nvidia-nemotron-3-reasoning-challenge")
TRAIN_PATH      = DATA_DIR / "train.csv"
TEST_PATH       = DATA_DIR / "test.csv"
OUTPUT_DIR      = "/kaggle/working/nemotron_lora_adapter"
SUBMISSION_PATH = "/kaggle/working/submission.csv"
METRIC_PATH     = "/kaggle/usr/lib/notebooks/metric/nvidia-nemotron-metric/metric.py"
LORA_RANK       = 22
MAX_LENGTH      = 1024
EPOCHS          = 2
LR              = 1e-4

# ── dataset ───────────────────────────────────────────────────────────────────
class ReasoningDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.rows, self.tok, self.max_len = df.reset_index(drop=True), tokenizer, max_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row    = self.rows.iloc[idx]
        prompt = str(row["prompt"]).strip()
        answer = str(row["answer"]).strip()
        if not (answer.startswith("\\boxed{") and answer.endswith("}")):
            answer = f"\\boxed{{{answer}}}"

        user  = f"Solve the problem. Return only the final answer inside \\boxed{{}}.\n\n{prompt}\n\nAnswer:"
        full  = f"{user} {answer}{self.tok.eos_token}"
        n_prompt = len(self.tok(user, truncation=True, max_length=512, add_special_tokens=True)["input_ids"])

        enc = self.tok(full, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
        ids, mask = enc["input_ids"][0], enc["attention_mask"][0]
        labels = ids.clone()
        labels[:n_prompt] = -100
        labels[mask == 0] = -100
        return {"input_ids": ids, "attention_mask": mask, "labels": labels}

# ── main ──────────────────────────────────────────────────────────────────────
def main():
    if LORA_RANK > 32:
        raise ValueError("LORA_RANK must be <= 32.")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16)
    model = get_peft_model(model, LoraConfig(
        r=LORA_RANK, lora_alpha=16,
        target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
        lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
    ))
    model.config.use_cache = False
    model.enable_input_require_grads() 
    model.print_trainable_parameters()

    df       = pd.read_csv(TRAIN_PATH).sample(frac=1.0, random_state=42).reset_index(drop=True)
    split    = int(len(df) * 0.80)
    train_df, eval_df = df.iloc[:split].copy(), df.iloc[split:].copy()

    try:
        import bitsandbytes; optim = "paged_adamw_8bit"
    except ImportError:
        optim = "adamw_torch"

    eval_key = "eval_strategy" if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames else "evaluation_strategy"
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir="/kaggle/working/nemotron_training_logs",
            logging_dir="/kaggle/working/logs",
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=1, per_device_eval_batch_size=1,
            gradient_accumulation_steps=5, learning_rate=LR,
            warmup_steps=50, max_grad_norm=1.0,
            bf16=True, optim=optim,
            logging_strategy="steps", logging_steps=10,
            save_strategy="steps", save_steps=500, save_total_limit=2,
            report_to="tensorboard",
            remove_unused_columns=False, gradient_checkpointing=True,
            **{eval_key: "steps"}, eval_steps=100,
        ),
        train_dataset=ReasoningDataset(train_df, tokenizer, MAX_LENGTH),
        eval_dataset =ReasoningDataset(eval_df,  tokenizer, MAX_LENGTH),
    )
    trainer.train()

    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Saved LoRA adapter to: {OUTPUT_DIR}")

    spec = importlib.util.spec_from_file_location("nvidia_nemotron_metric", METRIC_PATH)
    metric = importlib.util.module_from_spec(spec); spec.loader.exec_module(metric)
    metric.DATA_PATH = DATA_DIR
    metric.generate_standard_submission(OUTPUT_DIR)
    submission = pd.read_csv("submission.csv")
    submission.to_csv(SUBMISSION_PATH, index=False)
    print(f"Saved submission to: {SUBMISSION_PATH}")

    test_df = pd.read_csv(TEST_PATH)
    row_id_col = str(test_df.columns[0])
    if "answer" in test_df.columns:
        acc = metric.score(
            solution=test_df[[row_id_col, "answer"]],
            submission=submission,
            row_id_column_name=row_id_col,
            max_lora_rank=LORA_RANK, debug=True,
        )
        print(f"Test accuracy: {acc:.6f}")

if __name__ == "__main__":
    main()

NameError: name 'kagglehub' is not defined

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"]     = "expandable_segments:True"
os.environ["TRANSFORMERS_NO_TF"]          = "1"
os.environ["TRANSFORMERS_NO_FLAX"]        = "1"
os.environ["CUDA_VISIBLE_DEVICES"]        = "0"
os.environ["TRITON_PTXAS_PATH"]           = "/tmp/triton/backends/nvidia/bin/ptxas"
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas-blackwell"

import site, subprocess, sys, types, re, zipfile, json, importlib.util
from pathlib import Path

# ── copy nvidia utilities ─────────────────────────────────────────────────────
_util = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script"
if os.path.exists(_util):
    subprocess.run(f"tar --no-same-permissions -cf - -C {_util} . | tar -xf - -C /tmp", shell=True, check=False)
    subprocess.run("find /tmp/triton -type f | xargs chmod +x", shell=True, check=False)
    site.addsitedir(f"{_util}/nvidia_cutlass_dsl/python_packages/")
sys.path.insert(0, '/tmp')

# ── mamba_ssm stub fix ────────────────────────────────────────────────────────
for _mod_name in [
    'mamba_ssm.modules.mamba3',
    'mamba_ssm.ops.cute',
    'mamba_ssm.ops.cute.mamba3',
    'mamba_ssm.ops.cute.mamba3.mamba3_step_fn',
]:
    sys.modules[_mod_name] = types.ModuleType(_mod_name)
sys.modules['mamba_ssm.modules.mamba3'].Mamba3 = None

# ── imports ───────────────────────────────────────────────────────────────────
import kagglehub
import pandas as pd
import torch
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

# ── config ────────────────────────────────────────────────────────────────────
MODEL_PATH      = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
DATA_DIR        = Path("/kaggle/input/nvidia-nemotron-3-reasoning-challenge")
TRAIN_PATH      = DATA_DIR / "train.csv"
OUTPUT_DIR      = "/kaggle/working/sft_adapter"
SUBMISSION_ZIP  = "/kaggle/working/submission.zip"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

LORA_RANK  = 32
LORA_ALPHA = 32
MAX_LENGTH = 4096
EPOCHS     = 1
LR         = 1e-4
SEED       = 123

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

# ── dataset ───────────────────────────────────────────────────────────────────
class ReasoningDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.rows     = df.reset_index(drop=True)
        self.tok      = tokenizer
        self.max_len  = max_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row    = self.rows.iloc[idx]
        prompt = str(row["prompt"]).strip()
        answer = str(row["answer"]).strip()

        # avoid double boxing
        m = re.search(r'\\boxed\s*\{(.+)\}\s*$', answer)
        if m:
            answer = m.group(1).strip()

        user      = prompt + PROMPT_SUFFIX
        full      = f"{user} Therefore, \\boxed{{{answer}}}{self.tok.eos_token}"
        n_prompt  = len(self.tok(
            user, truncation=True, max_length=self.max_len,
            add_special_tokens=True
        )["input_ids"])

        enc   = self.tok(
            full, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        ids   = enc["input_ids"][0]
        mask  = enc["attention_mask"][0]
        labels = ids.clone()
        labels[:n_prompt] = -100   # mask prompt
        labels[mask == 0] = -100   # mask padding
        return {"input_ids": ids, "attention_mask": mask, "labels": labels}

# ── submission zip ────────────────────────────────────────────────────────────
def build_submission_zip(adapter_dir, zip_path, base_model_name):
    adapter_dir = Path(adapter_dir)
    config_path = adapter_dir / "adapter_config.json"

    with open(config_path) as f:
        cfg = json.load(f)
    cfg["base_model_name_or_path"] = base_model_name
    cfg["inference_mode"]          = True
    cfg["lora_dropout"]            = 0.0
    with open(config_path, "w") as f:
        json.dump(cfg, f, indent=2)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fname in ["adapter_config.json", "adapter_model.safetensors"]:
            fpath = adapter_dir / fname
            if fpath.exists():
                zf.write(str(fpath), fname)
                print(f"  Added {fname} ({fpath.stat().st_size/1024/1024:.1f} MB)")
            else:
                print(f"  WARNING: {fname} not found")

    print(f"submission.zip: {Path(zip_path).stat().st_size/1024/1024:.1f} MB")

# ── main ──────────────────────────────────────────────────────────────────────
def main():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # plain bf16 — confirmed working on this GPU
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )

    model = get_peft_model(model, LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ))
    model.enable_input_require_grads()
    model.config.use_cache = False
    model.print_trainable_parameters()

    # data
    df       = pd.read_csv(TRAIN_PATH).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    split    = int(len(df) * 0.9)
    train_df = df.iloc[:split].copy()
    eval_df  = df.iloc[split:].copy()

    train_dataset = ReasoningDataset(train_df, tokenizer, MAX_LENGTH)
    eval_dataset  = ReasoningDataset(eval_df,  tokenizer, MAX_LENGTH)

    # optimizer — use paged_adamw if bitsandbytes available, else adamw
    try:
        import bitsandbytes
        optim = "paged_adamw_8bit"
    except ImportError:
        optim = "adamw_torch"

    eval_key = "eval_strategy" if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames else "evaluation_strategy"

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir="/kaggle/working/train_logs",
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=8,
            learning_rate=LR,
            lr_scheduler_type="cosine",
            warmup_ratio=0.05,
            max_grad_norm=1.0,
            bf16=True,
            optim=optim,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            logging_steps=10,
            save_strategy="no",
            report_to="none",
            remove_unused_columns=False,
            dataloader_num_workers=2,
            seed=SEED,
            **{eval_key: "steps"},
            eval_steps=100,
        ),
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
    )

    print("Starting training...")
    trainer.train()

    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Adapter saved: {OUTPUT_DIR}")

    build_submission_zip(OUTPUT_DIR, SUBMISSION_ZIP, BASE_MODEL_NAME)
    print(f"Done! Submission ready: {SUBMISSION_ZIP}")

if __name__ == "__main__":
    main()

/tmp/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116
Starting training...


Step,Training Loss,Validation Loss
100,3.229872,0.457977
200,2.625994,0.404884
300,2.928567,0.376877
400,3.024423,0.353854
500,2.539634,0.334767
600,2.955028,0.314184
700,1.918358,0.304879
800,2.503332,0.298561
900,3.061856,0.294920
1000,2.255923,0.291427


In [6]:

inputs = tokenizer("QUESTION  : 89+76= ? ", return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tshape nameokens=50)
print(tokenizer.decode(output[0], skip_special_tokens=True))

QUESTION  : 89+76= ?  A) 135  B) 145  C) 155  D) 165

ANSWER : 165 "

Probably they want explanation? Or they posted the question. They


In [ ]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

In [2]:
print('Done.')
%load_ext tensorboard

Done.
The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
